# 02 — Feature Engineering: FD001

Continues from 01_eda.ipynb. Covers:
- Per-engine baseline normalization
- Rolling window features (mean, std)

See DECISIONS2.md for reasoning behind each choice made here.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import SENSOR_COLS
from src.data_loader import load_subset
from src.preprocessing import add_rul_to_train, add_rul_to_test

pd.set_option("display.max_columns", 40)

## Reload FD001 and redo the steps from Phase 1
(RUL labeling + dropping constant sensors) so this notebook is
self-contained and doesn't depend on variables still sitting in memory
from 01_eda.ipynb.

In [2]:
train, test, rul_truth = load_subset("FD001")
train = add_rul_to_train(train)
test = add_rul_to_test(test, rul_truth)

constant_sensors = ["sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19"]
train_reduced = train.drop(columns=constant_sensors)
test_reduced = test.drop(columns=constant_sensors)

print("train_reduced shape:", train_reduced.shape)
train_reduced.head()

train_reduced shape: (20631, 20)


,unit_number,time_in_cycles,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,125
1,1,2,0.0019,-0.0003,100.0,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,125
2,1,3,-0.0043,0.0003,100.0,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,125
3,1,4,0.0007,0.0000,100.0,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,125
4,1,5,-0.0019,-0.0002,100.0,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,125


## Step 1: Per-engine baseline normalization (sensor_14 first, as a test case)

In [3]:
# Step 1a: for every row, find out what THIS ENGINE's very first reading was
first_values = train_reduced.groupby("unit_number")["sensor_14"].transform("first")

# Step 1b: subtract that baseline from the current reading, for every row
train_reduced["sensor_14_normalized"] = train_reduced["sensor_14"] - first_values

# Verify: engine 1's first row should show 0.0, then drift as cycles increase
train_reduced[train_reduced["unit_number"] == 1][["time_in_cycles", "sensor_14", "sensor_14_normalized"]].head(10)

,time_in_cycles,sensor_14,sensor_14_normalized
0,1,8138.62,0.00
1,2,8131.49,-7.13
2,3,8133.23,-5.39
3,4,8133.83,-4.79
4,5,8133.80,-4.82
5,6,8132.85,-5.77
6,7,8132.32,-6.30
7,8,8131.07,-7.55
8,9,8125.69,-12.93
9,10,8129.38,-9.24


In [4]:
correlation_before = train_reduced[["sensor_14", "RUL"]].corr().iloc[0, 1]
correlation_after = train_reduced[["sensor_14_normalized", "RUL"]].corr().iloc[0, 1]

print("Before normalization:", correlation_before)
print("After normalization:", correlation_after)

Before normalization: -0.3697525579251205
After normalization: -0.39719754506085336


In [5]:
# average of the first 10 cycles per engine, instead of just cycle 1
baseline_avg = (
    train_reduced[train_reduced["time_in_cycles"] <= 10]
    .groupby("unit_number")["sensor_14"]
    .mean()
)

# map that per-engine average back onto every row
train_reduced["sensor_14_baseline_avg"] = train_reduced["unit_number"].map(baseline_avg)
train_reduced["sensor_14_normalized_v2"] = train_reduced["sensor_14"] - train_reduced["sensor_14_baseline_avg"]

correlation_v2 = train_reduced[["sensor_14_normalized_v2", "RUL"]].corr().iloc[0, 1]
print("After normalization (avg of first 10 cycles):", correlation_v2)

After normalization (avg of first 10 cycles): -0.4076745340218136


In [6]:
sensor_cols_remaining = [c for c in train_reduced.columns if c.startswith("sensor_")]

for sensor in sensor_cols_remaining:
    # average of this engine's first 10 cycles = the baseline for this sensor
    baseline_avg = (
        train_reduced[train_reduced["time_in_cycles"] <= 10]
        .groupby("unit_number")[sensor]
        .mean()
    )
    baseline_mapped = train_reduced["unit_number"].map(baseline_avg)
    train_reduced[f"{sensor}_norm"] = train_reduced[sensor] - baseline_mapped

print("New columns added:")
print([c for c in train_reduced.columns if c.endswith("_norm")])

New columns added:
['sensor_2_norm', 'sensor_3_norm', 'sensor_4_norm', 'sensor_7_norm', 'sensor_8_norm', 'sensor_9_norm', 'sensor_11_norm', 'sensor_12_norm', 'sensor_13_norm', 'sensor_14_norm', 'sensor_15_norm', 'sensor_17_norm', 'sensor_20_norm', 'sensor_21_norm', 'sensor_14_normalized_norm', 'sensor_14_baseline_avg_norm', 'sensor_14_normalized_v2_norm']


In [7]:
comparison = pd.DataFrame({
    "raw_correlation": train_reduced[sensor_cols_remaining].corrwith(train_reduced["RUL"]),
    "normalized_correlation": train_reduced[[f"{s}_norm" for s in sensor_cols_remaining]]
        .corrwith(train_reduced["RUL"])
        .rename(lambda c: c.replace("_norm", "")),
})
comparison["improved"] = comparison["normalized_correlation"].abs() > comparison["raw_correlation"].abs()
print(comparison)

                         raw_correlation  normalized_correlation  improved
sensor_11                      -0.775230               -0.819268      True
sensor_12                       0.748870                0.787186      True
sensor_13                      -0.624034               -0.685760      True
sensor_14                      -0.369753               -0.407675      True
sensor_14_baseline_avg          0.021618                     NaN     False
sensor_14_normalized           -0.397198                     NaN     False
sensor_14_normalized_v2        -0.407675                     NaN     False
sensor_14alized                      NaN               -0.407675     False
sensor_14alized_v2                   NaN               -0.407675     False
sensor_15                      -0.720858               -0.742568      True
sensor_17                      -0.680829               -0.678399     False
sensor_2                       -0.678458               -0.679339      True
sensor_20                

/opt/anaconda3/envs/aiml/lib/python3.11/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/anaconda3/envs/aiml/lib/python3.11/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [8]:
sensor_cols_remaining = [c for c in SENSOR_COLS if c not in constant_sensors]
print(sensor_cols_remaining)   # sanity check: should be exactly 14 sensor names, nothing else

['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [9]:
train_features = train_reduced[["unit_number", "time_in_cycles", "RUL"] + sensor_cols_remaining].copy()

for sensor in sensor_cols_remaining:
    baseline_avg = (
        train_features[train_features["time_in_cycles"] <= 10]
        .groupby("unit_number")[sensor]
        .mean()
    )
    baseline_mapped = train_features["unit_number"].map(baseline_avg)
    train_features[f"{sensor}_norm"] = train_features[sensor] - baseline_mapped

print(train_features.shape)
train_features.head()

(20631, 31)


,unit_number,time_in_cycles,RUL,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21,sensor_2_norm,sensor_3_norm,sensor_4_norm,sensor_7_norm,sensor_8_norm,sensor_9_norm,sensor_11_norm,sensor_12_norm,sensor_13_norm,sensor_14_norm,sensor_15_norm,sensor_17_norm,sensor_20_norm,sensor_21_norm
0,1,1,125,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190,-0.381,1.988,-0.24,0.264,0.011,-3.37,0.198,-0.486,-0.024,6.392,0.01111,0.2,0.076,0.0294
1,1,2,125,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236,-0.051,4.108,2.30,-0.346,-0.009,-5.49,0.218,0.134,0.026,-0.738,0.02341,0.2,0.016,0.0340
2,1,3,125,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442,0.149,0.278,3.36,0.164,0.031,3.38,-0.002,0.274,-0.014,1.002,0.00941,-1.8,-0.034,-0.0454
3,1,4,125,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739,0.149,-4.922,1.03,0.354,0.061,-0.08,-0.142,0.714,0.036,1.602,-0.04019,0.2,-0.104,-0.0157
4,1,5,125,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044,0.169,-4.862,5.38,-0.096,0.011,5.59,0.008,0.044,-0.004,1.572,0.02101,1.2,-0.084,0.0148


In [10]:
comparison = pd.DataFrame({
    "raw_correlation": train_features[sensor_cols_remaining].corrwith(train_features["RUL"]),
    "normalized_correlation": train_features[[f"{s}_norm" for s in sensor_cols_remaining]]
        .corrwith(train_features["RUL"])
        .set_axis(sensor_cols_remaining),
})
comparison["improved"] = comparison["normalized_correlation"].abs() > comparison["raw_correlation"].abs()
print(comparison)

           raw_correlation  normalized_correlation  improved
sensor_2         -0.678458               -0.679339      True
sensor_3         -0.655030               -0.652631     False
sensor_4         -0.757157               -0.791568      True
sensor_7          0.733021                0.758788      True
sensor_8         -0.624568               -0.683860      True
sensor_9         -0.462151               -0.494433      True
sensor_11        -0.775230               -0.819268      True
sensor_12         0.748870                0.787186      True
sensor_13        -0.624034               -0.685760      True
sensor_14        -0.369753               -0.407675      True
sensor_15        -0.720858               -0.742568      True
sensor_17        -0.680829               -0.678399     False
sensor_20         0.704626                0.719492      True
sensor_21         0.707334                0.728244      True


In [11]:
redundancy_check = pd.DataFrame({
    "correlation_raw_vs_normalized": [
        train_features[sensor].corr(train_features[f"{sensor}_norm"])
        for sensor in sensor_cols_remaining
    ]
}, index=sensor_cols_remaining)

print(redundancy_check.sort_values("correlation_raw_vs_normalized"))

           correlation_raw_vs_normalized
sensor_13                       0.765619
sensor_8                        0.767169
sensor_12                       0.816521
sensor_7                        0.828941
sensor_11                       0.835460
sensor_2                        0.867718
sensor_20                       0.868202
sensor_4                        0.869215
sensor_15                       0.872474
sensor_21                       0.873676
sensor_17                       0.880460
sensor_3                        0.913572
sensor_14                       0.923356
sensor_9                        0.942859


In [12]:
window_sizes = [5, 10, 15, 20]

for window in window_sizes:
    for sensor in sensor_cols_remaining:
        for suffix in ["", "_norm"]:          # do this for both raw and normalized versions
            col = f"{sensor}{suffix}"
            new_col = f"{col}_rollmean_{window}"

            train_features[new_col] = (
                train_features.groupby("unit_number")[col]
                .rolling(window=window, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)   # fix the index mismatch described above
            )

print("Total columns now:", train_features.shape[1])


Total columns now: 143


/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/2253475671.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[new_col] = (
/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/2253475671.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[new_col] = (
/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/2253475671.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

In [13]:
results = []
for window in window_sizes:
    for sensor in sensor_cols_remaining:
        col = f"{sensor}_rollmean_{window}"
        corr = train_features[col].corr(train_features["RUL"])
        results.append({"sensor": sensor, "window": window, "correlation": corr})

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index="sensor", columns="window", values="correlation")
print(pivot)

window           5         10        15        20
sensor                                           
sensor_11 -0.814046 -0.808694 -0.798614 -0.787102
sensor_12  0.793414  0.788734  0.778602  0.766826
sensor_13 -0.661564 -0.653075 -0.639985 -0.625812
sensor_14 -0.371717 -0.368984 -0.365712 -0.362002
sensor_15 -0.810432 -0.814061 -0.807167 -0.797384
sensor_17 -0.803477 -0.815216 -0.812173 -0.804587
sensor_2  -0.794731 -0.803831 -0.798426 -0.789091
sensor_20  0.802195  0.807712  0.801641  0.792372
sensor_21  0.804467  0.809448  0.803193  0.793955
sensor_3  -0.795402 -0.811018 -0.809692 -0.803385
sensor_4  -0.815743 -0.814031 -0.805780 -0.795519
sensor_7   0.792644  0.789922  0.780465  0.769102
sensor_8  -0.662449 -0.653610 -0.640225 -0.625702
sensor_9  -0.467569 -0.466264 -0.464258 -0.461709


In [14]:
results_norm = []
for window in [5, 10]:
    for sensor in sensor_cols_remaining:
        col = f"{sensor}_norm_rollmean_{window}"
        corr = train_features[col].corr(train_features["RUL"])
        results_norm.append({"sensor": sensor, "window": window, "correlation": corr})

results_norm_df = pd.DataFrame(results_norm)
pivot_norm = results_norm_df.pivot(index="sensor", columns="window", values="correlation")
print(pivot_norm)

window           5         10
sensor                       
sensor_11 -0.877705 -0.883099
sensor_12  0.852300  0.858442
sensor_13 -0.756944 -0.763336
sensor_14 -0.414260 -0.416307
sensor_15 -0.855147 -0.869281
sensor_17 -0.813236 -0.831504
sensor_2  -0.813905 -0.832272
sensor_20  0.838864  0.854905
sensor_21  0.849304  0.865147
sensor_3  -0.808224 -0.831662
sensor_4  -0.872260 -0.880994
sensor_7   0.839608  0.847220
sensor_8  -0.755449 -0.761033
sensor_9  -0.504309 -0.507391


In [15]:
for window in [5, 10]:
    for sensor in sensor_cols_remaining:
        for suffix in ["", "_norm"]:
            col = f"{sensor}{suffix}"
            new_col = f"{col}_rollstd_{window}"

            train_features[new_col] = (
                train_features.groupby("unit_number")[col]
                .rolling(window=window, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
            )

print("Total columns now:", train_features.shape[1])

Total columns now: 199


/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/3922284795.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[new_col] = (
/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/3922284795.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_features[new_col] = (
/var/folders/5w/40v6y__153s245ll58v9krdw0000gn/T/ipykernel_46130/3922284795.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

In [16]:
# how many NaNs did rollstd introduce, and where?
rollstd_cols = [c for c in train_features.columns if "_rollstd_" in c]
print(train_features[rollstd_cols].isnull().sum())

# confirm it's happening at the very start of each engine's life
train_features[train_features["unit_number"] == 1][["time_in_cycles"] + rollstd_cols[:2]].head(3)

sensor_2_rollstd_5           100
sensor_2_norm_rollstd_5      100
sensor_3_rollstd_5           100
sensor_3_norm_rollstd_5      100
sensor_4_rollstd_5           100
sensor_4_norm_rollstd_5      100
sensor_7_rollstd_5           100
sensor_7_norm_rollstd_5      100
sensor_8_rollstd_5           100
sensor_8_norm_rollstd_5      100
sensor_9_rollstd_5           100
sensor_9_norm_rollstd_5      100
sensor_11_rollstd_5          100
sensor_11_norm_rollstd_5     100
sensor_12_rollstd_5          100
sensor_12_norm_rollstd_5     100
sensor_13_rollstd_5          100
sensor_13_norm_rollstd_5     100
sensor_14_rollstd_5          100
sensor_14_norm_rollstd_5     100
sensor_15_rollstd_5          100
sensor_15_norm_rollstd_5     100
sensor_17_rollstd_5          100
sensor_17_norm_rollstd_5     100
sensor_20_rollstd_5          100
sensor_20_norm_rollstd_5     100
sensor_21_rollstd_5          100
sensor_21_norm_rollstd_5     100
sensor_2_rollstd_10          100
sensor_2_norm_rollstd_10     100
sensor_3_r

,time_in_cycles,sensor_2_rollstd_5,sensor_2_norm_rollstd_5
0,1,NaN,NaN
1,2,0.233345,0.233345
2,3,0.267644,0.267644


In [17]:
for col in rollstd_cols:
    train_features[col] = train_features[col].fillna(0)

# verify no NaNs remain
print(train_features[rollstd_cols].isnull().sum().sum())   # should print 0

0


In [18]:
results_std = []
for window in [5, 10]:
    for sensor in sensor_cols_remaining:
        for suffix in ["", "_norm"]:
            col = f"{sensor}{suffix}_rollstd_{window}"
            corr = train_features[col].corr(train_features["RUL"])
            results_std.append({"sensor": f"{sensor}{suffix}", "window": window, "correlation": corr})

results_std_df = pd.DataFrame(results_std)
pivot_std = results_std_df.pivot(index="sensor", columns="window", values="correlation")
print(pivot_std.sort_values(5))

window                5         10
sensor                            
sensor_14      -0.105053 -0.299263
sensor_14_norm -0.105053 -0.299263
sensor_9       -0.096873 -0.282487
sensor_9_norm  -0.096873 -0.282487
sensor_13      -0.053791 -0.105606
sensor_13_norm -0.053791 -0.105606
sensor_11      -0.043620 -0.114738
sensor_11_norm -0.043620 -0.114738
sensor_17      -0.041975 -0.079294
sensor_17_norm -0.041975 -0.079294
sensor_12      -0.041415 -0.105055
sensor_12_norm -0.041415 -0.105055
sensor_4       -0.029066 -0.082175
sensor_4_norm  -0.029066 -0.082175
sensor_8       -0.028290 -0.067021
sensor_8_norm  -0.028290 -0.067021
sensor_7       -0.022406 -0.064781
sensor_7_norm  -0.022406 -0.064781
sensor_20      -0.020835 -0.058199
sensor_20_norm -0.020835 -0.058199
sensor_15_norm -0.015029 -0.050266
sensor_15      -0.015029 -0.050266
sensor_3       -0.014836 -0.038504
sensor_3_norm  -0.014836 -0.038504
sensor_2       -0.012337 -0.041137
sensor_2_norm  -0.012337 -0.041137
sensor_21_norm  0.00

In [19]:
# drop the redundant normalized rollstd columns — mathematically identical to raw rollstd
duplicate_std_cols = [c for c in train_features.columns if "_norm_rollstd_" in c]
train_features = train_features.drop(columns=duplicate_std_cols)

print("Dropped:", len(duplicate_std_cols), "duplicate columns")
print("Total columns now:", train_features.shape[1])

Dropped: 28 duplicate columns
Total columns now: 171


In [20]:
# confirm no _norm_rollstd_ columns remain
remaining_norm_rollstd = [c for c in train_features.columns if "_norm_rollstd_" in c]
print(remaining_norm_rollstd)   # should print an empty list []

[]


In [21]:
def engineer_features(df, sensor_cols, windows=[5, 10]):
    """
    Applies baseline normalization, rolling mean, and rolling std
    to a CMAPSS dataframe (train OR test) — entirely per-engine,
    so it's safe to call independently on each.
    """
    df = df[["unit_number", "time_in_cycles", "RUL"] + sensor_cols].copy()

    # Step 1: baseline normalization (per-engine, own first 10 cycles)
    for sensor in sensor_cols:
        baseline_avg = (
            df[df["time_in_cycles"] <= 10]
            .groupby("unit_number")[sensor]
            .mean()
        )
        baseline_mapped = df["unit_number"].map(baseline_avg)
        df[f"{sensor}_norm"] = df[sensor] - baseline_mapped

    # Step 2: rolling mean (raw + normalized)
    for window in windows:
        for sensor in sensor_cols:
            for suffix in ["", "_norm"]:
                col = f"{sensor}{suffix}"
                new_col = f"{col}_rollmean_{window}"
                df[new_col] = (
                    df.groupby("unit_number")[col]
                    .rolling(window=window, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                )

    # Step 3: rolling std (raw only — normalized version is mathematically
    # identical, confirmed in DECISIONS2.md entry 6)
    for window in windows:
        for sensor in sensor_cols:
            col = sensor
            new_col = f"{col}_rollstd_{window}"
            df[new_col] = (
                df.groupby("unit_number")[col]
                .rolling(window=window, min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

    return df

In [22]:
train_final = engineer_features(train_reduced, sensor_cols_remaining)
test_final = engineer_features(test_reduced, sensor_cols_remaining)

print("train_final:", train_final.shape)
print("test_final:", test_final.shape)

assert train_final.shape[1] == test_final.shape[1], "Column count mismatch between train and test!"
print("Column counts match:", train_final.shape[1])

train_final: (20631, 115)
test_final: (13096, 115)
Column counts match: 115


In [23]:
print(test_final["RUL"].describe())
print(test_final["RUL"].isnull().sum())   # should be 0

count    13096.000000
mean       108.919823
std         27.580258
min          7.000000
25%        102.000000
50%        125.000000
75%        125.000000
max        125.000000
Name: RUL, dtype: float64
0


In [24]:
import numpy as np

np.random.seed(42)   # so the split is reproducible every time you rerun this
all_engine_ids = train_final["unit_number"].unique()
np.random.shuffle(all_engine_ids)

n_val_engines = 20
val_engine_ids = all_engine_ids[:n_val_engines]
train_engine_ids = all_engine_ids[n_val_engines:]

train_split = train_final[train_final["unit_number"].isin(train_engine_ids)]
val_split = train_final[train_final["unit_number"].isin(val_engine_ids)]

print("Train engines:", len(train_engine_ids), "-> rows:", train_split.shape[0])
print("Val engines:", len(val_engine_ids), "-> rows:", val_split.shape[0])

Train engines: 80 -> rows: 16561
Val engines: 20 -> rows: 4070
